In [2]:
!pip install gpxpy


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [14]:
tour_name = "Hemelse Hike 2025"
folder = "/Users/ernestjanssen/_Projects/Zwerfbond/Gpx routes Hemelse Hike 2025/"
gpx_files = ["HeH 2025 do.gpx","HeH 2025 vr.gpx","HeH 2025 za-versie maart.gpx","HeH 2025 zo-via Ferme Libert.gpx"]
days = ["dag 1","dag 2","dag 3","dag 4"]
xyz_to_mbtiles_folder = "/Users/ernestjanssen/_Projects/Zwerfbond/xyz-to-mbtiles-master/"
mbtiles_output = "hemelse_hike_output.mbtiles"
mbtiles_file = xyz_to_mbtiles_folder+mbtiles_output

In [15]:
preview_data = open(folder+gpx_files[0]).read()
print(preview_data)

<?xml version="1.0"?>
<gpx xmlns="http://www.topografix.com/GPX/1/1" version="1.1" creator="afstandmeten.nl" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.topografix.com/GPX/1/1 http://www.topografix.com/GPX/1/1/gpx.xsd">
  <trk>
    <name>HeH 2025 do</name>
    <desc/>
    <trkseg>
      <trkpt lat="50.5187721" lon="6.0633223">
        <ele>0.00000</ele>
        <time>2024-11-17T00:00:00Z</time>
      </trkpt>
      <trkpt lat="50.518453" lon="6.0638995">
        <ele>0.00000</ele>
        <time>2024-11-17T00:00:21Z</time>
      </trkpt>
      <trkpt lat="50.5183221" lon="6.0635992">
        <ele>0.00000</ele>
        <time>2024-11-17T00:00:30Z</time>
      </trkpt>
      <trkpt lat="50.5182075" lon="6.0632087">
        <ele>0.00000</ele>
        <time>2024-11-17T00:00:42Z</time>
      </trkpt>
      <trkpt lat="50.5181543" lon="6.0630799">
        <ele>0.00000</ele>
        <time>2024-11-17T00:00:46Z</time>
      </trkpt>
      <trkpt lat="50.51

In [16]:
## CHECK OF 2 OPEENVOLGENDE PUNTEN DEZELFDE LAT EN LON COORDINATEN HEBBEN 
## --> DIT KAN LEIDEN TOT EINDIGE RODE STREPEN OP UITEINDELIJKE ROUTEKAART




In [17]:
import gpxpy
import gpxpy.gpx

route_boundaries = [0]*4 #[min_lon, max_lon, min_lat, max_lat]
route_coordinates = {}
i=0
for index,gpx_file in enumerate(gpx_files):
    gpx_file = open(folder+gpx_file,'r')
    gpx = gpxpy.parse(gpx_file)
    coordinates_list = []
    for track in gpx.tracks:
        for segment in track.segments:
            for point in segment.points:
                if i==0:
                    route_boundaries = [point.longitude,point.longitude,point.latitude,point.latitude]
                    i+=1
                coordinates_list.append(point.longitude)
                coordinates_list.append(point.latitude)
                if route_boundaries[0]>point.longitude:
                    route_boundaries[0]=point.longitude
                if route_boundaries[1]<point.longitude:
                    route_boundaries[1]=point.longitude
                if route_boundaries[2]>point.latitude:
                    route_boundaries[2]=point.latitude
                if route_boundaries[3]<point.latitude:
                    route_boundaries[3]=point.latitude      
                
    route_coordinates[days[index]] = coordinates_list

# print(route_coordinates)
# print(route_boundaries)
map_boundaries = [0]*4 #[min_lon, max_lon, min_lat, max_lat]
map_boundaries[0] = route_boundaries[0]-0.02
map_boundaries[1] = route_boundaries[2]-0.02
map_boundaries[2] = route_boundaries[1]+0.02
map_boundaries[3] = route_boundaries[3]+0.02
print(map_boundaries)

# map_boundaries_sorted = [0]*4 #[min_lon, min_lat, max_lon, max_lat]
# map_boundaries_sorted[0] = map_boundaries[0]
# map_boundaries_sorted[1] = map_boundaries[2]
# map_boundaries_sorted[2] = map_boundaries[1]
# map_boundaries_sorted[3] = map_boundaries[3]
# print(map_boundaries)


[6.009693, 50.414424, 6.1535553, 50.539030000000004]


**--------------------------**
**Download nu de OpenTopo kaart met bovenstaande bbox met het xyz-to-mbtiles nodeJS programma!**
**--------------------------**

Voer onderstaand command uit in de xyz-to-mbtiles-master folder uit:

node index.js --input "https://tile.opentopomap.org/{z}/{x}/{y}.png" --output "mbtiles_output" --header 'Referer:https://opentopomap.org/' --header "Origin:https://opentopomap.org/" --retry 4 --minzoom 12 --maxzoom 16 --bbox "map_boundaries"



In [18]:
## Create route_coordinates table in mbtiles database

import sqlite3

db_mbtiles = sqlite3.connect(mbtiles_file)
c_mbtiles = db_mbtiles.cursor()
c_mbtiles.execute("CREATE TABLE IF NOT EXISTS `route_coordinates` (`day` varchar(20) NOT NULL, `coordinate_string` blob NOT NULL)")

for day in days:
    sql = "INSERT INTO route_coordinates (day,coordinate_string) VALUES (?,?)"
    val = (day,bytes(str(route_coordinates[day]),'utf-8'))
    c_mbtiles.execute(sql,val)
    
c_mbtiles.close()
db_mbtiles.commit()
db_mbtiles.close()

In [19]:
##Define tour name in metadata of mbtiles database; f.e. "Paastocht 2022" of "Hemelse Hike 2022"

db_mbtiles = sqlite3.connect(mbtiles_file)
c_mbtiles = db_mbtiles.cursor()
sql = f"UPDATE metadata SET value = '{tour_name}' WHERE name = ?"
value = ('name',)
c_mbtiles.execute(sql,value)
db_mbtiles.commit()
print("Total number of rows updated :", db_mbtiles.total_changes)
myresult = c_mbtiles.fetchall()
db_mbtiles.close()
print(myresult)

Total number of rows updated : 1
[]


In [20]:
##Define the days of the tour in the metadata of the mbtiles database

days_sql = str()
for day in days:
    days_sql+=day + ", "
days_sql = days_sql.rstrip(", ")

db_mbtiles = sqlite3.connect(mbtiles_file)
c_mbtiles = db_mbtiles.cursor()
sql = "INSERT INTO metadata (name,value) VALUES ('days', ?);"
value = (days_sql,)
c_mbtiles.execute(sql,value)
db_mbtiles.commit()
myresult = c_mbtiles.fetchall()
db_mbtiles.close()

IntegrityError: UNIQUE constraint failed: metadata.name

In [ ]:
**--------------------------**
**Hernoem het bestand naar map_data.mbtiles.**
**Zet het klaar op de zwerfbond server.**
**--------------------------**

ssh ernest@64.227.116.105
exit  
scp /Users/ernestjanssen/_Projects/Zwerfbond/APP/map_data.mbtiles ernest@64.227.116.105:/home/ernest
